In [ ]:
from dataclasses import dataclass, field
from collections import defaultdict
import numpy as np

@dataclass
class Component:

    name: str
    systems: list = field(default_factory=list)
    parents: list = field(default_factory=list)
    bigparents: list = field(default_factory=list)
    sizes: list = field(default_factory=list)
    roles: list = field(default_factory=list)
    data: object = None
    parameters: dict = None
    true_parameters: dict = field(default_factory=lambda: {
        "mtbf_lambda": np.random.uniform(100, 10000),
        "mttr_lambda": np.random.uniform(80, 8000),
        "spare_rate": np.random.uniform(0.1, 1.25),
        "unit_price": np.random.uniform(100, 10000),
        "important": np.random.choice([True, False], p=[0.8, 0.2]),
    })


@dataclass
class System:

    name: str
    components: list = field(default_factory=list)
    parents: list = field(default_factory=list)
    bigparents: list = field(default_factory=list)
    size: str = None
    roles: list = field(default_factory=list)
    data: object = None
    parameters: dict = None
    true_parameters: dict = field(default_factory=lambda: {
        "say_hio": (np.random.randint(70, 100),np.random.randint(1, 30)),
        "speed": np.random.uniform(1, 8),
        "efficiency": np.random.uniform(0.5, 1.0)
    })

    def create_maintenance_record(self, horizon=100000):
        log = []
        t = 0
        while t < horizon:
            part = np.random.choice(self.components)
            ttf = np.random.exponential(part.true_parameters["mtbf_lambda"])
            ttr = np.random.exponential(part.true_parameters["mttr_lambda"])
            failure_time = t + ttf
            repair_complete = failure_time + ttr
            log.append((t,failure_time,repair_complete,part))
            t = repair_complete
        return log


@dataclass
class Parent:

    name: str
    systems: list = field(default_factory=list)
    components: list = field(default_factory=list)
    bigparent: object = None
    sizes: list = field(default_factory=list)
    roles: list = field(default_factory=list)
    data: object = None
    parameters: dict = None
    true_parameters: dict = None


@dataclass
class BigParent:

    name: str
    parents: list = field(default_factory=list)
    systems: list = field(default_factory=list)
    components: list = field(default_factory=list)
    sizes: list = field(default_factory=list)
    roles: list = field(default_factory=list)
    data: object = None
    parameters: dict = None
    true_parameters: dict = None


class Topology:

    def __init__(self,comp_sys_map,sys_parent_map,parent_bigparent_map,sys_size_map,parent_role_map):
        self.comp_sys_map = comp_sys_map
        self.sys_parent_map = sys_parent_map
        self.parent_bigparent_map = parent_bigparent_map
        self.sys_size_map = sys_size_map
        self.parent_role_map = parent_role_map
        self._build_reverse_maps()
        self._build_objects()
        self._connect_objects()

    def _build_reverse_maps(self):

        self.sys_comp_map = defaultdict(list)

        for comp, systems in self.comp_sys_map.items():
            for sys in systems:
                self.sys_comp_map[sys].append(comp)

        self.parent_sys_map = defaultdict(list)

        for sys, parents in self.sys_parent_map.items():
            for parent in parents:
                self.parent_sys_map[parent].append(sys)

        self.bigparent_parent_map = defaultdict(list)

        for parent, bigparent in self.parent_bigparent_map.items():
            self.bigparent_parent_map[bigparent].append(parent)

    def _build_objects(self):
        self.components = {name: Component(name)for name in self.comp_sys_map}
        self.systems = {name: System(name) for name in self.sys_parent_map}
        self.parents = {name: Parent(name) for name in self.parent_bigparent_map}
        self.bigparents = {name: BigParent(name) for name in self.bigparent_parent_map}

    def _connect_objects(self):

        for sys_name, system in self.systems.items():
            system.components = [self.components[c] for c in self.sys_comp_map[sys_name]]
            system.parents = [self.parents[p] for p in self.sys_parent_map[sys_name]]
            system.bigparents = list({self.bigparents[self.parent_bigparent_map[p.name]]for p in system.parents})
            system.size = self.sys_size_map[sys_name]
            system.roles = sorted({
                role
                for parent in system.parents
                for role in self.parent_role_map[parent.name]
            })



        for comp_name, component in self.components.items():

            component.systems = [
                self.systems[s]
                for s in self.comp_sys_map[comp_name]
            ]

            component.parents = list({
                parent
                for sys in component.systems
                for parent in sys.parents
            })

            component.bigparents = list({
                bp
                for parent in component.parents
                for bp in [parent.bigparent]
                if bp is not None
            })

            component.sizes = sorted({
                sys.size
                for sys in component.systems
            })

            component.roles = sorted({
                role
                for parent in component.parents
                for role in parent.roles
            })

        # ---------- parents ----------

        for parent_name, parent in self.parents.items():

            parent.systems = [
                self.systems[s]
                for s in self.parent_sys_map[parent_name]
            ]

            parent.components = list({
                comp
                for sys in parent.systems
                for comp in sys.components
            })

            parent.bigparent = self.bigparents[
                self.parent_bigparent_map[parent_name]
            ]

            parent.sizes = sorted({
                sys.size
                for sys in parent.systems
            })

            parent.roles = sorted(
                set(self.parent_role_map[parent_name])
            )

        # ---------- bigparents ----------

        for bp_name, bp in self.bigparents.items():

            bp.parents = [
                self.parents[p]
                for p in self.bigparent_parent_map[bp_name]
            ]

            bp.systems = list({
                sys
                for parent in bp.parents
                for sys in parent.systems
            })

            bp.components = list({
                comp
                for sys in bp.systems
                for comp in sys.components
            })

            bp.sizes = sorted({
                sys.size
                for sys in bp.systems
            })

            bp.roles = sorted({
                role
                for parent in bp.parents
                for role in parent.roles
            })

    # --------------------------------------------------------

    @classmethod
    def random(cls, seed=123):

        np.random.seed(seed)

        comp_list = [f"comp_{i}" for i in range(1, 20)]
        sys_list = [f"sys_{i}" for i in range(1, 15)]
        parent_list = [f"parent_{i}" for i in range(1, 10)]
        bigparent_list = [f"bigparent_{i}" for i in range(1, 5)]

        size_list = ["S", "M", "L", "XL"]
        roles = ["Admin", "User", "Guest", "Superuser"]

        comp_sys_map = {
            comp: list(np.random.choice(
                sys_list,
                size=np.random.randint(1, 6),
                replace=False
            ))
            for comp in comp_list
        }

        sys_parent_map = {
            sys: list(np.random.choice(
                parent_list,
                size=np.random.randint(1, 8),
                replace=False
            ))
            for sys in sys_list
        }

        parent_bigparent_map = {
            parent: np.random.choice(bigparent_list)
            for parent in parent_list
        }

        sys_size_map = {
            sys: np.random.choice(size_list)
            for sys in sys_list
        }

        parent_role_map = {
            parent: list(np.random.choice(
                roles,
                size=np.random.randint(1, 5),
                replace=False
            ))
            for parent in parent_list
        }

        return cls(
            comp_sys_map,
            sys_parent_map,
            parent_bigparent_map,
            sys_size_map,
            parent_role_map
        )